# Notebook 03 — ColPali Visual RAG

This is the main event. Notebook 02 ended with CLIP failing to find *"Q3 revenue"* on a chart, because a 512-d vector for a whole page is too lossy. **ColPali fixes that.**

## The idea (S3 §4.3)

Traditional document RAG: `parse → OCR → chunk → embed text → search`. Layout, charts, tables, equations get destroyed in the OCR step.

ColPali (Faysse et al., 2024) skips OCR entirely:

1. Render each PDF page **as an image**.
2. Run it through a VLM (PaliGemma) and keep **all ~1024 patch embeddings** — not a single pooled vector.
3. At query time, embed the query into N token vectors.
4. Score with **late interaction (MaxSim)**: for each query token, take the max similarity against any patch on the page; sum the maxes. Pages where specific patches match specific query tokens win.

```
Standard CLIP-style:                    ColPali (late interaction):
page → [single 1024-d vector]           page  → [1024 patch vectors]
query → [single 1024-d vector]          query → [N token vectors]
score = cos(page_vec, query_vec)        score = sum_t max_p (q_t · p)
```

This preserves visual structure. "Q3 revenue" → the query token `"Q3"` directly matches the *bar labelled Q3*; `"revenue"` matches the *Y-axis label*. We'll see it work on a real PDF.

## Hardware

ColPali (~3 B params with the SigLIP-PaliGemma backbone) needs ~6–10 GB of memory in bf16. Apple Silicon with 16 GB RAM works via MPS. NVIDIA with ≥10 GB VRAM works via CUDA. CPU is too slow to be practical.

If your machine can't run ColPali, skim this notebook and run Notebook 04 — Notebook 04 doesn't depend on a working ColPali index.

## 1. Setup

Drop a PDF (a financial report or a paper with figures works best) into `../data/sample_pdfs/`. We'll use the first PDF we find.

In [ ]:
from pathlib import Path

PDF_DIR = Path("../data/sample_pdfs")
pdfs = sorted(PDF_DIR.glob("*.pdf"))
assert pdfs, f"Drop at least one PDF into {PDF_DIR.resolve()}"
PDF_PATH = pdfs[0]
print(f"Using: {PDF_PATH.name}")

In [ ]:
import torch
from transformers import ColPaliForRetrieval, ColPaliProcessor

MODEL_NAME = "vidore/colpali-v1.3-hf"
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {device}")

model = ColPaliForRetrieval.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=device,
).eval()
processor = ColPaliProcessor.from_pretrained(MODEL_NAME)
print("ColPali loaded.")

## 2. Render PDF pages as images

`pdf2image` rasterizes each page. DPI 150 is a good balance — high enough that small chart text is legible, low enough that rendering and embedding stay fast.

In [ ]:
from pdf2image import convert_from_path

pages = convert_from_path(str(PDF_PATH), dpi=150)
print(f"Rendered {len(pages)} pages.")

# Preview the first 3
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, min(3, len(pages)), figsize=(12, 5))
if len(pages) == 1:
    axes = [axes]
for ax, p, i in zip(axes, pages[:3], range(3)):
    ax.imshow(p); ax.set_title(f"Page {i}"); ax.set_axis_off()
plt.tight_layout(); plt.show()

## 3. Embed every page

ColPali outputs one **(num_patches, 128)** tensor per page. The 128-d patch vectors are L2-normalized — handy because dot product = cosine.

Batch size matters: ColPali is memory-heavy (~1 GB per page in flight). Use `batch_size=4` on a 16 GB machine. Drop to 1 if you OOM.

In [ ]:
@torch.no_grad()
def embed_pages(pil_pages, batch_size: int = 4):
    """Returns a list of (num_patches, 128) bf16 CPU tensors, one per page."""
    out = []
    for i in range(0, len(pil_pages), batch_size):
        batch = pil_pages[i : i + batch_size]
        inputs = processor(images=batch, return_tensors="pt").to(device)
        embeddings = model(**inputs).embeddings   # (B, num_patches, 128)
        for emb in embeddings:
            out.append(emb.cpu())
        print(f"  embedded {i + len(batch)}/{len(pil_pages)} pages")
    return out

page_embeddings = embed_pages(pages)
print(f"\nFirst page tensor shape: {page_embeddings[0].shape}")
print(f"Total stored vectors: {sum(t.shape[0] for t in page_embeddings)}")

Notice the size — a few thousand patch vectors per document. This is the storage tradeoff (S3 §4.3 table): ColPali uses ~1000× more storage per page than CLIP, in exchange for state-of-the-art quality on text-heavy PDFs. Production systems mitigate with **token pooling** (HierarchicalTokenPooler) and **int8 quantization** (Vespa).

## 4. Embed a query and score with MaxSim

The text path produces an (N_query_tokens, 128) tensor. Late interaction:

```
for each query token t:
    max_sim = max over patches of (q_t · patch)
page_score = sum of those max_sims
```

One matmul + a max + a sum. The `processor.score_retrieval` helper bundles this, but writing it ourselves is more educational.

In [ ]:
@torch.no_grad()
def embed_query(query: str) -> torch.Tensor:
    """Returns a (num_query_tokens, 128) tensor."""
    inputs = processor(text=[query], return_tensors="pt").to(device)
    return model(**inputs).embeddings[0].cpu()


def maxsim(query_emb: torch.Tensor, page_emb: torch.Tensor) -> float:
    """Sum over query tokens of the max patch similarity."""
    # query_emb: (T, 128)   page_emb: (P, 128)
    sim = query_emb.float() @ page_emb.float().T   # (T, P)
    return float(sim.max(dim=1).values.sum())


def search_pages(query: str, k: int = 3):
    q = embed_query(query)
    scored = [(maxsim(q, pe), idx) for idx, pe in enumerate(page_embeddings)]
    scored.sort(reverse=True)
    return scored[:k]


# Tweak this query to something specific to your PDF
QUERY = "summary of key findings"
hits = search_pages(QUERY, k=3)
for score, idx in hits:
    print(f"page {idx}  score={score:.2f}")

In [ ]:
# Visualize the top hit
top_score, top_idx = hits[0]
fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(pages[top_idx])
ax.set_title(f"Top match for: '{QUERY}'\npage {top_idx} · MaxSim={top_score:.2f}")
ax.set_axis_off()
plt.show()

## 5. Try the queries that broke CLIP

Adapt these to whatever's actually in your PDF — the point is to ask things that need *layout-aware* matching:

In [ ]:
# Replace these with queries appropriate for your PDF
DEMO_QUERIES = [
    "revenue chart",
    "table of results",
    "introduction",
]

fig, axes = plt.subplots(1, len(DEMO_QUERIES), figsize=(5 * len(DEMO_QUERIES), 6))
if len(DEMO_QUERIES) == 1:
    axes = [axes]
for ax, q in zip(axes, DEMO_QUERIES):
    s, i = search_pages(q, k=1)[0]
    ax.imshow(pages[i])
    ax.set_title(f"'{q}' → page {i} (score {s:.1f})", fontsize=10)
    ax.set_axis_off()
plt.tight_layout(); plt.show()

## 6. Generate the answer with a VLM

ColPali only **retrieves**. It doesn't generate. The full RAG pipeline is:

```
query → ColPali → top-K page images → VLM → grounded answer
```

We send the top page as an image to GPT-4o. The system prompt forces the model to answer **only from the page** and to cite the page number — that's the cheap, immediate hallucination control. We'll do the heavier-duty schema-and-evidence version in Notebook 04.

Make sure your `OPENAI_API_KEY` is set in `.env`.

In [ ]:
import base64, io, os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path="../.env")
client = OpenAI()

def page_to_b64(page_image, quality: int = 85) -> str:
    buf = io.BytesIO()
    page_image.save(buf, format="JPEG", quality=quality)
    return base64.b64encode(buf.getvalue()).decode()


SYSTEM = (
    "You answer questions using ONLY the provided PDF page images. "
    "Always cite the page number you used. "
    "If the answer isn't visible on the provided pages, say so explicitly."
)

def visual_rag(query: str, top_k: int = 3, model_name: str = "gpt-4o-mini"):
    hits = search_pages(query, k=top_k)
    image_parts = []
    used_pages = []
    for score, idx in hits:
        used_pages.append(idx)
        image_parts.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{page_to_b64(pages[idx])}",
                "detail": "high",   # text-heavy → high
            },
        })
    user_text = f"Pages provided (in order): {used_pages}\n\nQuestion: {query}"
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": [*image_parts, {"type": "text", "text": user_text}]},
        ],
    )
    return {
        "answer": response.choices[0].message.content,
        "used_pages": used_pages,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
    }


result = visual_rag(QUERY, top_k=2)
print(f"Used pages: {result['used_pages']}")
print(f"Tokens — input: {result['input_tokens']}, output: {result['output_tokens']}")
print("\n" + result["answer"])

Try harder questions. ColPali shines when the answer lives in a **specific patch** (a chart bar, a table cell, a figure caption) rather than in flowing prose.

## What we built

End-to-end Visual RAG:

```
PDF → render pages → ColPali embed (1024 vectors/page)
                         ↓
                   in-memory store
                         ↓
user question → ColPali query embed → MaxSim → top-K pages
                                         ↓
                       GPT-4o w/ page images → cited answer
```

What this would look like in production:
- **Vector store:** Milvus 2.6+ (`Array of Structs` for one-row-per-page) or Vespa with int8 quantization.
- **Two-stage retrieval:** dense single-vector recall (top-100), then ColPali MaxSim rerank (top-5). Cuts MaxSim work by 100×.
- **Token pooling:** HierarchicalTokenPooler — 66% fewer patches at <2% quality loss. Free win.
- **Caching:** rendered page images cached on S3, keyed by `(pdf_hash, page_num, dpi)`.

**Next:** [Notebook 04 — Grounded generation](04_grounded_generation.ipynb). The retrieval is solved. Now we need to make the *generation* step refuse to hallucinate.